In [ ]:
import kagglehub

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)
os.listdir(path)


In [ ]:
# Task 1: Write your code here:

import kagglehub
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

file_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(file_path)

df.head()











In [ ]:
# Task 2: Write your code here:
print("First 5 rows of the dataset:")
display(df.head())

In [ ]:
# Task 3: Write your code here:
print("\nDataset info:")
df.info()


In [ ]:
# Task 4: Write your code here:
print("\nStatistical description:")
display(df.describe())


In [ ]:
# Task 5: Write your code here:

plt.figure(figsize=(6,4))
plt.hist(df["Delivery_Time"], bins=30)
plt.title("Delivery Time Distribution")
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.show()



In [ ]:
# Task 1: Write your code here:
if "Order_ID" in df.columns:
    df = df.drop(columns=["Order_ID"])


In [ ]:
# Task 2: Write your code here:
# Check missing values
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(exclude=["number"]).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [ ]:
# Task 3: Write your code here:

dup_count = df.duplicated().sum()
print("Number of duplicate rows:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed. New shape:", df.shape)
else:
    print("No duplicate samples found.")


In [ ]:
# Task 4: Write your code here:
df = pd.get_dummies(df, drop_first=True)


In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler
import pandas as pd

target_col = "Delivery_Time"

X = df.drop(columns=[target_col])
y = df[target_col]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X = pd.DataFrame(X_scaled, columns=X.columns)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
# Task 6: Write your code here:

target_counts = y.value_counts()
print("Target counts:")
print(target_counts)

print("\nTarget proportions:")
print((target_counts / target_counts.sum()).round(4))

target_counts.plot(kind="bar", title="Target Distribution")
plt.show()


In [ ]:
# Task 1: Write your code here:
target_col = "Delivery_Time"

X = df.drop(columns=[target_col])
y = df[target_col]



In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# T2

if y.nunique() <= 20:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    splits = cv.split(X, y)
else:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = cv.split(X)

mae_scores = []

# T3,4
for train_idx, val_idx in splits:
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    fold_mae = mean_absolute_error(y_val, preds)
    mae_scores.append(fold_mae)

# T5
print("Fold MAEs:", np.round(mae_scores, 4))
print("Average MAE:", round(float(np.mean(mae_scores)), 4))


In [ ]:
# Task 1: Write your code here:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

final_model = RandomForestRegressor(random_state=42)
final_model.fit(X, y)

fi = pd.DataFrame({
    "feature": X.columns,
    "importance": final_model.feature_importances_
}).sort_values("importance", ascending=False)

print(fi.head(10))

fi.head(15).plot(kind="barh", x="feature", y="importance", title="Top Feature Importances")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
import matplotlib.pyplot as plt

preds = final_model.predict(X)

plt.hist(preds, bins=25)
plt.title("Predicted Delivery Time Histogram")
plt.xlabel("Predicted delivery time")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Task Bonus: Write your code here:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

try:
    from catboost import CatBoostRegressor
    catboost_ok = True
except:
    catboost_ok = False
    print("CatBoost not available - will run RandomForest only.")

target_col = "Delivery_Time"
X = df.drop(columns=[target_col])
y = df[target_col]

# KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: RandomForest
    rf = RandomForestRegressor(random_state=42, n_estimators=300)
    rf.fit(X_train, y_train)
    pred_rf = rf.predict(X_val)

    # Model 2: CatBoost (if available)
    if catboost_ok:
        cb = CatBoostRegressor(random_state=42, verbose=0)
        cb.fit(X_train, y_train)
        pred_cb = cb.predict(X_val)

        # Avp
        pred_avg = (pred_rf + pred_cb) / 2
    else:
        pred_avg = pred_rf

    mae = mean_absolute_error(y_val, pred_avg)
    mae_scores.append(mae)
    print(f"Fold {fold} MAE:", round(mae, 4))

print("\nAverage MAE across folds:", round(float(np.mean(mae_scores)), 4))

